# Chapter 28 — Growth Models and Simulation

Companion code for *Python for Indian Economics Students*. The explanations, exercises and discussion are in the book.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/slxnjdhv/python-for-indian-economics/blob/main/labs/ch28_growth_models.ipynb)

In [ ]:
# Run this cell first on Google Colab (not needed on your own computer
# if you installed requirements.txt).
import sys
if 'google.colab' in sys.modules:
    !pip install -q git+https://github.com/slxnjdhv/python-for-indian-economics.git

## 28.1 The Penn World Table for India

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize

from indiadata import pwt

india = pwt("IND").set_index("year")
cols = ["rgdpna", "rnna", "rkna", "emp", "hc", "labsh", "delta", "rtfpna",
        "csh_i"]
india = india[cols].astype(float)
print("years:", india.index.min(), "to", india.index.max())
india.loc[[1950, 1991, 2019], ["rgdpna", "rnna", "emp", "hc", "labsh", "delta"]].round(3)

In [ ]:
print(india["labsh"].tail(5).round(4).to_string())

## 28.2 Growth accounting

In [ ]:
logs = np.log(india[["rgdpna", "rnna", "emp", "hc"]])
d = logs.diff()
labour_share = (india["labsh"] + india["labsh"].shift(1)) / 2
alpha_t = 1 - labour_share

ga = pd.DataFrame({
    "GDP growth": d["rgdpna"],
    "capital": alpha_t * d["rnna"],
    "employment": labour_share * d["emp"],
    "human capital": labour_share * d["hc"],
})
ga["TFP"] = ga["GDP growth"] - ga[["capital", "employment", "human capital"]].sum(axis=1)
ga = ga.dropna()

decades = pd.cut(ga.index, bins=[1950, 1960, 1970, 1980, 1990, 2000, 2010, 2019],
                 labels=["1951-60", "1961-70", "1971-80", "1981-90",
                         "1991-2000", "2001-10", "2011-19"])
by_decade = 100 * ga.groupby(decades, observed=True).mean()
by_decade.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))
parts = ["capital", "employment", "human capital", "TFP"]
colours = ["#1f4e79", "#9db7d5", "#7f7f7f", "#c55a11"]
x = np.arange(len(by_decade))
pos = np.zeros(len(x))
neg = np.zeros(len(x))
for part, colour in zip(parts, colours):
    v = by_decade[part].to_numpy()
    bottom = np.where(v >= 0, pos, neg)
    ax.bar(x, v, bottom=bottom, color=colour, width=0.65, label=part)
    pos += np.where(v >= 0, v, 0)
    neg += np.where(v < 0, v, 0)
ax.plot(x, by_decade["GDP growth"], "o", color="black", label="GDP growth")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x, by_decade.index)
ax.set_ylabel("percentage points per year")
ax.set_title("Sources of India's growth by decade", loc="left", fontsize=13)
ax.legend(frameon=False, ncol=5, loc="upper left", fontsize=9)
ax.set_ylim(min(neg.min(), 0) - 0.5, pos.max() + 1.2)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "Average annual log growth. Source: Penn World Table "
         "10.01 (Feenstra, Inklaar and Timmer 2015); own growth accounting "
         "with Törnqvist labour-share weights.", fontsize=8, color="grey")
plt.show()

### Check: our TFP against PWT's

In [ ]:
def tfp_index(capital):
    resid = (d["rgdpna"] - alpha_t * np.log(india[capital]).diff()
             - labour_share * (d["emp"] + d["hc"]))
    idx = np.exp(resid.dropna().cumsum())
    return idx / idx.loc[2017]


cmp = pd.DataFrame({"with capital stock (rnna)": tfp_index("rnna"),
                    "with capital services (rkna)": tfp_index("rkna"),
                    "PWT rtfpna": india["rtfpna"]}).dropna()
for col in cmp.columns[:2]:
    gap = (cmp[col] - cmp["PWT rtfpna"]).abs().max()
    corr = np.log(cmp[[col, "PWT rtfpna"]]).diff().corr().iloc[0, 1]
    print(f"{col}: largest gap {gap:.3f}, growth correlation {corr:.3f}")
cmp.loc[[1960, 1980, 1991, 2000, 2010, 2019]].round(3)

## 28.3 The Solow model as a difference equation

In [ ]:
def solow_path(k0, s, alpha, delta, n, g, T):
    """Capital per effective worker for T periods, starting at k0."""
    k = np.empty(T + 1)
    k[0] = k0
    for t in range(T):
        k[t + 1] = (s * k[t] ** alpha + (1 - delta) * k[t]) / ((1 + n) * (1 + g))
    return k


def gdp_growth(k, alpha, n, g):
    """Annual growth of total GDP along a path of k."""
    y = k ** alpha
    return (y[1:] / y[:-1]) * (1 + n) * (1 + g) - 1


def k_steady(s, alpha, delta, n, g):
    return (s / ((1 + n) * (1 + g) - (1 - delta))) ** (1 / (1 - alpha))

### Calibrating to India

In [ ]:
from indiadata import mospi_nas, NAS_REVISIONS

RANK = {r: i for i, r in enumerate(NAS_REVISIONS)}


def nas_total(indicator):
    """All-India annual total, latest estimate of each year (as in Chapter 9)."""
    t = mospi_nas(indicator)
    if "institutional_sector" in t.columns:
        t = t[t["institutional_sector"].isna()
              | t["institutional_sector"].str.startswith("Total", na=False)]
    t = (t.assign(rank=t["revision"].map(RANK).fillna(-1))
          .sort_values(["year", "rank"]).drop_duplicates("year", keep="last"))
    return t.set_index("year")[["current_price", "constant_price", "revision"]]


gdp_nas = nas_total("GDP")
gfcf_nas = nas_total("GFCF")
inv = pd.DataFrame({
    "GFCF/GDP current, %": 100 * gfcf_nas["current_price"] / gdp_nas["current_price"],
    "GFCF/GDP constant, %": 100 * gfcf_nas["constant_price"] / gdp_nas["constant_price"],
    "real GDP growth, %": 100 * gdp_nas["constant_price"].pct_change(),
    "estimate": gdp_nas["revision"]}).dropna(subset=["GFCF/GDP current, %"])
inv.tail(4).round(2)

In [ ]:
s_nas = inv.loc["2023-24":"2025-26", "GFCF/GDP current, %"].mean() / 100
window = slice(2010, 2019)
alpha = 1 - india.loc[window, "labsh"].mean()
delta = india.loc[2019, "delta"]
n = np.exp((d["emp"] + d["hc"]).loc[window].mean()) - 1
g_tfp = ga.loc[window, "TFP"].mean()
g = np.exp(g_tfp / (1 - alpha)) - 1
KY_2019 = india.loc[2019, "rnna"] / india.loc[2019, "rgdpna"]

calib = pd.Series({
    "s (GFCF/GDP current prices, NAS, 2023-24 to 2025-26)": s_nas,
    "alpha (1 - labour share, 2010-19)": alpha,
    "delta (PWT, 2019)": delta,
    "n (growth of emp x hc, 2010-19)": n,
    "TFP growth (2010-19)": np.exp(g_tfp) - 1,
    "g (labour-augmenting, = TFP/(1-alpha))": g,
    "K/Y in 2019 (PWT)": KY_2019,
})
calib.round(4)

In [ ]:
k_star_nas = k_steady(s_nas, alpha, delta, n, g)
print(f"steady-state K/Y with s = {s_nas:.3f}: {k_star_nas ** (1 - alpha):.2f}"
      f"   (India in 2019: {KY_2019:.2f})")

In [ ]:
s_implied = ((india["rnna"].shift(-1) - (1 - india["delta"]) * india["rnna"])
             / india["rgdpna"])
s_pwt = s_implied.loc[2010:2018].mean()
compare_s = pd.Series({
    "NAS GFCF/GDP, current prices, 2023-24 to 2025-26": s_nas,
    "NAS GFCF/GDP, constant 2011-12 prices, same years":
        inv.loc["2023-24":"2025-26", "GFCF/GDP constant, %"].mean() / 100,
    "PWT csh_i (current PPP prices), 2019": india.loc[2019, "csh_i"],
    "implied by PWT capital stock, 2010-2018 average": s_pwt,
})
print(compare_s.round(3).to_string())
k_star = k_steady(s_pwt, alpha, delta, n, g)
KY_star = k_star ** (1 - alpha)
ss_growth = (1 + n) * (1 + g) - 1
print(f"\nsteady-state K/Y with s = {s_pwt:.3f}: {KY_star:.2f}   "
      f"(India in 2019: {KY_2019:.2f})")
print(f"steady-state GDP growth: {ss_growth:.2%}   "
      f"(effective labour n = {n:.2%}, technology g = {g:.2%})")

### Simulating the transition

In [ ]:
k0 = KY_2019 ** (1 / (1 - alpha))
T = 60
path = solow_path(k0, s_pwt, alpha, delta, n, g, T)
growth = gdp_growth(path, alpha, n, g)
print(f"model GDP growth: {growth[0]:.2%} in 2020, "
      f"{growth[:10].mean():.2%} average 2020-2029, "
      f"{growth[-1]:.2%} after {T} years")

# A permanent rise in the investment rate by 5 points, from the 2019 position
shock = solow_path(k0, s_pwt + 0.05, alpha, delta, n, g, T)
gr_shock = gdp_growth(shock, alpha, n, g)
k_new = k_steady(s_pwt + 0.05, alpha, delta, n, g)
half = np.argmax(np.abs(shock - k_new) < 0.5 * abs(k0 - k_new))
print(f"with s 5 points higher: growth {gr_shock[0]:.2%} in 2020, "
      f"{gr_shock[:10].mean():.2%} average 2020-29; "
      f"half the gap to the new steady state closed after {half} years")

## 28.4 The golden rule

In [ ]:
c_star = lambda s: (1 - s) * k_steady(s, alpha, delta, n, g) ** alpha
gold = optimize.minimize_scalar(lambda s: -c_star(s), bounds=(0.01, 0.99),
                                method="bounded")
print(f"golden-rule saving rate: {gold.x:.4f}   (alpha = {alpha:.4f})")
print(f"steady-state consumption, golden rule / at s = {s_pwt:.3f}: "
      f"{c_star(gold.x) / c_star(s_pwt):.4f}")

## 28.5 What would 8 per cent growth take?

In [ ]:
TARGET = 0.08


def avg_growth_10y(s, g_=g):
    k = solow_path(k0, s, alpha, delta, n, g_, 10)
    return gdp_growth(k, alpha, n, g_).mean()


rows = []
for s_try in (0.40, s_pwt, 0.50, 0.55, 0.60, 0.65):
    rows.append({"investment rate (PWT units)": s_try,
                 "avg growth 2020-29, %": 100 * avg_growth_10y(s_try)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

s_needed = optimize.brentq(lambda s: avg_growth_10y(s) - TARGET, 0.05, 0.95)
path_8 = gdp_growth(solow_path(k0, s_needed, alpha, delta, n, g, 40), alpha, n, g)
print(f"\ninvestment rate needed for 8% average growth in 2020-29: {s_needed:.1%}"
      f" in PWT units")
print(f"   = {s_needed - s_pwt:+.1%} points above the 2010-18 rate; roughly "
      f"{s_nas * s_needed / s_pwt:.0%} in National Accounts terms (scaled)")
print(f"   growth on that path: {path_8[9]:.2%} in 2029, {path_8[-1]:.2%} in 2059")

In [ ]:
g_needed = optimize.brentq(lambda gg: (1 + n) * (1 + gg) - 1 - TARGET, 0.0, 0.2)
tfp_needed = (1 + g_needed) ** (1 - alpha) - 1
print(f"labour-augmenting growth needed for 8% steady state: {g_needed:.2%}")
print(f"   = TFP growth of {tfp_needed:.2%} a year "
      f"(2010-2019 average: {np.exp(g_tfp) - 1:.2%})")
roll = ga["TFP"].rolling(10).mean()
print(f"highest 10-year average TFP growth, 1951-2019: "
      f"{np.exp(roll.max()) - 1:.2%} (ten years to {roll.idxmax()})")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.6))
for s_try, colour, lw in ((s_pwt, "#1f4e79", 2.2), (s_pwt + 0.05, "#9db7d5", 1.8),
                          (s_needed, "#c55a11", 2.2)):
    gr = gdp_growth(solow_path(k0, s_try, alpha, delta, n, g, 40), alpha, n, g)
    ax.plot(np.arange(2020, 2060), 100 * gr, color=colour, linewidth=lw,
            label=f"investment rate {s_try:.1%} (PWT units)")
ax.axhline(8, color="#c55a11", linestyle=":", linewidth=1)
ax.text(2059, 8.1, "8%", ha="right", fontsize=9, color="#c55a11")
ax.set_ylabel("model GDP growth, % per year")
ax.set_title("Higher investment buys faster growth for a while, not forever",
             loc="left", fontsize=12)
ax.legend(frameon=False, loc="upper right")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.07, 1, 1))
fig.text(0.01, 0.005, "Solow model simulated from India's 2019 capital-output "
         "ratio; alpha, delta, n, g and the investment rate implied by the "
         "capital stock\nfrom Penn World Table 10.01 (2010-19). A simulation, "
         "not a forecast; every path returns to the same steady-state growth.",
         fontsize=8, color="grey")
plt.show()

## 28.6 Choosing saving optimally: value function iteration

In [ ]:
BETA = 0.95                                    # illustrative assumption
k_ss = (alpha * BETA) ** (1 / (1 - alpha))     # analytic steady state
kgrid = np.linspace(0.2 * k_ss, 2.0 * k_ss, 150)

# consumption for every (today k, tomorrow k') pair; -inf if infeasible
cons = kgrid[:, None] ** alpha - kgrid[None, :]
util = np.where(cons > 0, np.log(np.where(cons > 0, cons, 1.0)), -np.inf)

V = np.zeros(len(kgrid))
errors = []
for it in range(1000):
    V_new = (util + BETA * V[None, :]).max(axis=1)
    err = np.max(np.abs(V_new - V))
    errors.append(err)
    V = V_new
    if err < 1e-8:
        break
policy = kgrid[(util + BETA * V[None, :]).argmax(axis=1)]
print(f"VFI converged after {it + 1} iterations (sup-norm change {err:.1e})")
print(f"memory of the choice table: {util.nbytes / 1e3:.0f} kB")

analytic = alpha * BETA * kgrid ** alpha
step = kgrid[1] - kgrid[0]
print(f"largest gap between VFI and analytic policy: "
      f"{np.max(np.abs(policy - analytic)):.4f} (grid step {step:.4f})")
print(f"implied saving rate k'/f(k): VFI {np.median(policy / kgrid ** alpha):.4f}, "
      f"analytic alpha*beta = {alpha * BETA:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3))
ax1.plot(kgrid, policy, color="#1f4e79", linewidth=2, label="VFI on a grid")
ax1.plot(kgrid, analytic, color="#c55a11", linestyle="--", linewidth=1.5,
         label="analytic: k' = αβk^α")
ax1.plot(kgrid, kgrid, color="grey", linewidth=0.8, label="45° line")
ax1.set_xlabel("capital today, k")
ax1.set_ylabel("capital tomorrow, k'")
ax1.set_title("Optimal saving policy", loc="left", fontsize=11)
ax1.legend(frameon=False, fontsize=9)
ax2.semilogy(np.arange(1, len(errors) + 1), errors, color="#1f4e79",
             label="value function iteration")
ax2.semilogy(np.arange(1, len(errors) + 1),
             errors[0] * BETA ** np.arange(len(errors)), color="#c55a11",
             linestyle="--", label="β^t (contraction bound)")
ax2.set_xlabel("iteration")
ax2.set_ylabel("max |V_new − V| (log scale)")
ax2.set_title("Value function iteration converges", loc="left", fontsize=11)
ax2.legend(frameon=False, fontsize=9)
for a in (ax1, ax2):
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.01, 0.005, "Brock-Mirman model: log utility, full depreciation; "
         "alpha calibrated from PWT, beta = 0.95 is an illustrative "
         "assumption. 150-point grid.", fontsize=8, color="grey")
plt.show()

## 28.7 Answering the question

## Key ideas

## Exercises

## Data citation

## Further reading